# Visualizations

> Data visualization

In [ ]:
#| default_exp visualize

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

# =================================
# Standard libraries
# =================================
import base64
import io

# =================================
# Scientific / data
# =================================
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde

# =================================
# Visualization
# =================================
import matplotlib.pyplot as plt
from contextlib import nullcontext
from matplotlib.ticker import MaxNLocator
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# =================================
# PyTorch / vision
# =================================
import torchvision

# =================================
# fastai
# =================================
from fastai.vision.all import ifnone

# =================================
# bonsai
# =================================
from bonsai.utils import *
from bonsai.datasets import cells3d

In [ ]:
#| hide
pio.renderers.default = "notebook_connected"

In [ ]:

# load sample data
data4D = img2float(cells3d())
data = data4D[:, 1, :, :] # load the nuclei channel

## Images

### Display 2D images

Function to quickly display 2D images

In [ ]:
#| export
@delegates(plt.Axes.imshow, keep=True)
def show_image(values, # A 2D or 3D array of pixel values representing the image.
               cmap = 'gray', # The colormap to use for displaying the image. Default is 'gray' for grayscale images.
               **kwargs,
               ):
    """
    Show a 2D image or the central slice of a 3D image using Matplotlib. 
    The function assumes that 'values' is a 2D or 3D array representing an image, typically in grayscale.
    """
    # Check if the input is 3D
    if values.ndim == 3:
        # Get the central slice along the first axis
        central_slice = values[values.shape[0] // 2]
    else:
        central_slice = values
    
    # Create a figure with Matplotlib
    fig = plt.figure()
    
    # Add an axis to the figure, which will be used for plotting the image
    ax = fig.add_subplot(111)
    
    # Display the image using imshow. The np.flipud function is applied to flip the array vertically, 
    # which is necessary because Matplotlib's default orientation might differ from NumPy's or Plotly's.
    ax.imshow(np.flipud(central_slice), cmap=cmap, **kwargs)
    
    # Turn off axis numbers and ticks for a cleaner visual appearance of the image
    ax.axis('off')
    
    # Display the figure with all defined elements
    plt.show()

In [ ]:
# Example usage:
show_image(data[25])


When a 3D image is given, show_image will show the central slice.

In [ ]:
show_image(data)

### Display Multichannel images

Function to display RGB and multichannel images

In [ ]:
#| export
@delegates(plt.Axes.imshow, keep=True, but=['shape', 'imlim'])
def show_multichannel(
    img,                       # A tensor or numpy array representing a multi-channel image.
    ax=None,                   # The Matplotlib axis to use for plotting.
    figsize=None,              # The size of the figure.
    title=None,                # The title of the image.
    max_slices=3,              # The maximum number of slices to display.
    ctx=None,                  # The context to use for plotting.
    layout='horizontal',       # The layout type: 'horizontal', 'square', or 'multirow'.
    num_cols=3,                # Number of columns for the 'multirow' layout.
    channel_axis=0,            # Channel axis. 0 for channel-first, -1 for channel-last.
    **kwargs,
):
    """
    Show a multi-channel image.

    By default, images are assumed to be channel-first (C,Y,X). Use
    ``channel_axis=-1`` for channel-last (Y,X,C) images.
    """

    # Convert to NumPy
    if hasattr(img, 'as_tensor'):
        img_tensor = img.as_tensor().cpu().numpy()
    else:
        try:
            img_tensor = np.asarray(img)
        except Exception as e:
            raise TypeError(
                "The provided image cannot be converted to a tensor or numpy array."
            ) from e

    if img_tensor.ndim != 3:
        raise ValueError(
            f"Expected a 3D multi-channel image, got shape {img_tensor.shape}."
        )

    # Normalize to channel-first internally
    if channel_axis < 0:
        channel_axis += img_tensor.ndim

    if channel_axis not in range(img_tensor.ndim):
        raise ValueError(
            f"Invalid channel_axis={channel_axis} for image with "
            f"{img_tensor.ndim} dimensions."
        )

    if channel_axis != 0:
        img_tensor = np.moveaxis(img_tensor, channel_axis, 0)

    num_channels = img_tensor.shape[0]

    # Set figure size if not provided
    if figsize is None:
        figsize = (5, 5) if num_channels == 3 else (8, 4)

    # Create axis if not provided
    ax = ifnone(ax, ctx)
    if ax is None:
        _, ax = plt.subplots(figsize=figsize)

    # RGB
    if num_channels == 3:
        img_to_show = img_tensor.transpose(1, 2, 0)

        if np.max(img_to_show) > 1.0:
            img_to_show = img_to_show / 255

        ax.imshow(img_to_show, **kwargs)

    # Multi-channel grayscale
    else:
        # Limit number of channels to max_slices
        if num_channels > max_slices:
            start = num_channels // 2 - max_slices // 2
            end = start + max_slices
            img_tensor = img_tensor[start:end]
            num_channels = img_tensor.shape[0]

        if layout == 'square':
            grid_size = int(np.ceil(np.sqrt(num_channels)))
            height, width = img_tensor.shape[1:3]

            grid_canvas = np.zeros(
                (grid_size * height, grid_size * width)
            )

            for idx, channel in enumerate(img_tensor):
                row = idx // grid_size
                col = idx % grid_size

                grid_canvas[
                    row * height:(row + 1) * height,
                    col * width:(col + 1) * width,
                ] = channel

            img_to_show = grid_canvas

        elif layout == 'multirow':
            num_rows = int(np.ceil(num_channels / num_cols))
            height, width = img_tensor.shape[1:3]

            multirow_canvas = np.zeros(
                (num_rows * height, num_cols * width)
            )

            for idx, channel in enumerate(img_tensor):
                row = idx // num_cols
                col = idx % num_cols

                multirow_canvas[
                    row * height:(row + 1) * height,
                    col * width:(col + 1) * width,
                ] = channel

            img_to_show = multirow_canvas

        else:
            img_to_show = np.concatenate(
                [img_tensor[c] for c in range(num_channels)],
                axis=1,
            )

        ax.imshow(img_to_show, **kwargs)

    if title:
        ax.set_title(title)

    ax.axis('off')

    plt.show()

    return ax

In [ ]:
print(data4D[35].shape)
show_multichannel(data4D[35], cmap='gray', layout='multirow', num_cols=1);

### Display 3D images

Function to display 3D images

In [ ]:
#| export
# Utility function for determining figure bounds
def _fig_bounds(x):
    r = x//8
    return min(15, max(1,r))

In [ ]:
#| export
@delegates(plt.Axes.imshow, keep=True)
def show_mosaic(t: np.ndarray | torchTensor,        # 3D image to plot
                  axis: int = 0,                    # axis to split 3D array to 2D images
                  figsize: tuple = (15,15),         # size of the figure
                  cmap: str = 'gray',               # colormap to use
                  nrow: int = 10,                   # number of images per row
                  alpha = 1.,                       # transparency of the image
                  return_grid = False,              # return the grid for further processing
                  add_to_existing = False,          # add to existing figure
                  **kwargs):
    '''
    Plots 2D slices of a 3D image alongside a prior specified axis.
    '''
    if isinstance(t, np.ndarray): 
        t = tensor(t)
    
    if t.device.type == 'cuda': 
        t = t.cpu()
    
    if t.ndim < 3 or t.ndim > 4:
         raise TypeError('Object is not a rank 3 tensor but a rank {} tensor'.format(t.ndim ))
    if t.ndim == 4:
        t = t[0]

    if axis == 1: 
        t = np.transpose(t, (1, 0, 2))
    elif axis == 2: 
        t = np.transpose(t, (2, 0, 1))

    if t.min() < 0.:
        t = t - t.min()  # shift pixel values to positive range, otherwise the negative values will be clipped.

    t = t.unsqueeze(1)/t.max()  # rescale the images, makes for a nicer plot
    grid = torchvision.utils.make_grid(t, nrow=nrow)

    if return_grid:
        return grid

    if figsize is None:
        figsize = (_fig_bounds(t.shape[2]), _fig_bounds(t.shape[3]))

    if not add_to_existing:
        plt.figure(figsize=figsize)

    plt.imshow(grid[0,:,:], cmap=cmap, alpha=alpha, **kwargs)
    plt.axis('off')

In [ ]:
#| export
@delegates(plt.Axes.imshow, keep=True, but=['shape', 'imlim'])
def show_images_grid(images,            # A list of images to display.
                     ax=None,           # The Matplotlib axis to use for plotting.
                     ncols=10,          # The number of columns in the grid.
                     figsize=None,      # The size of the figure.
                     title=None,        # The title of the image.
                     spacing=0.02,      # The spacing between subplots.
                     max_slices=3,      # The maximum number of slices to display.
                     ctx=None,          # The context to use for plotting.
                     **kwargs):
    """
    Show a list of images arranged in a grid.

    Returns:
    - axes: matplotlib axes containing the grid of images.
    """

    # Ensure images is a list
    images = list(images)

    # Compute number of rows needed based on number of columns and images
    n_images = len(images)
    nrows = (n_images - 1) // ncols + 1

    # Compute figure size if not provided
    if figsize is None:
        fig_width = ncols * (1 + spacing) - spacing
        fig_height = nrows * (1 + spacing) - spacing
        figsize = (fig_width, fig_height)

    ax = ifnone(ax,ctx)
    
    if ax is None:
        fig, ax = plt.subplots(nrows, ncols, figsize=figsize)
        fig.subplots_adjust(wspace=spacing, hspace=spacing) # Adjust spacing between subplots
    ax = np.atleast_2d(ax)

    # Turn off axes for all subplots
    for axis in ax.flatten():
        axis.axis('off')

    # Iterate over images and corresponding axes
    for i, (im, axis) in enumerate(zip(images, ax.flatten())):
        # Convert input to numpy array if necessary
        if isinstance(im, np.ndarray):
            im_to_show = im
        elif hasattrs(im, ('data', 'cpu', 'permute')):
            im_to_show = im.data.cpu()
            if im_to_show.ndim == 3:
                num_slices = im_to_show.shape[0]
                if num_slices < max_slices:
                    im_to_show = np.concatenate([im_to_show[i] for i in range(num_slices)], axis=1)
                else:
                    # showing middle slice
                    im_to_show = im_to_show[num_slices // 2]
            elif im_to_show.ndim == 4:
                im_to_show = im_to_show[0]
        else:
            im_to_show = array(im)

        # Display image on current axes
        axis.imshow(im_to_show, **kwargs)
        if title is not None and i < len(title):
            axis.set_title(title[i])

    # Hide unused axes
    for axis in ax.flatten()[n_images:]:
        axis.set_visible(False)

    return ax

In [ ]:
show_mosaic(torch_from_numpy(data), figsize=None)

In [ ]:
show_images_grid(data, cmap='gray');

### Display Slices

In [ ]:
#| export
def show_plane(ax,                  # The axis object to display the slice on.
               plane,               # A 2D numpy array representing the slice of the image tensor.
               cmap="gray",         # Colormap to use for displaying the image.
               title=None,          # Title for the plot.
               lines=None,          # A list of indices where dashed lines should be drawn on the plane.
               linestyle='--',      # The style of the dashed lines.
               linecolor='white',   # The color of the dashed lines.
               ):
    """
    Display a slice of the image tensor on a given axis with optional dashed lines.
    """
    ax.imshow(plane, cmap=cmap)
    ax.set_axis_off()
    
    if title:
        ax.set_title(title)
    
    # Add dashed lines if specified
    if lines is not None:
        ax.axhline(lines[0], color=linecolor, linestyle=linestyle)
        ax.axvline(lines[1], color=linecolor, linestyle=linestyle)


In [ ]:
#| export
@delegates(show_plane, keep=True)
def show_slices(data,               # A 3D numpy array representing the image tensor.
                planes=None,        # A tuple containing the indices of the planes to visualize.
                showlines=True,     # Whether to show dashed lines on the planes, rows, and columns.
                style='default',    # style options
                **kwargs):
    """
    Visualize slices of a 3D image tensor along its planes, rows, and columns.
    """
    if planes:
        z, y, x = planes
    else:
        z, y, x = data.shape
        z, y, x = z // 2, y // 2, x // 2

    with plt.style.context(style or 'default'):
        _, (a, b, c) = plt.subplots(ncols=3, figsize=(15, 5))

        if showlines:
            show_plane(a, data[z], title=f'Plane = {z}', lines=[y, x], **kwargs)
            show_plane(b, data[:, y, :], title=f'Row = {y}', lines=[z, x], **kwargs)
            show_plane(c, data[:, :, x], title=f'Column = {x}', lines=[z, y], **kwargs)
        else:
            show_plane(a, data[z], title=f'Plane = {z}', **kwargs)
            show_plane(b, data[:, y, :], title=f'Row = {y}', **kwargs)
            show_plane(c, data[:, :, x], title=f'Column = {x}', **kwargs)

        plt.show()

In [ ]:
show_slices(data, showlines=False)

In [ ]:
show_slices(data, (25,100,150), linestyle=':', style="dark_background")

In [ ]:
#| export
def slice_explorer(data,            # A 3D numpy array representing the image tensor.
                   order='CZYX',    # The order of dimensions in the data.
                   **kwargs):
    """
    Visualizes the provided data using Plotly's interactive imshow function with animation support.
    """

    # Handle 4D data with channel dimension
    if len(data.shape) == 4:
        target_order = 'ZCYX'
        # If data is already in 'ZCYX' order, no need to transpose
        if order != target_order:
            # Create a mapping from the current order to the target order
            current_order_indices = [order.index(dim) for dim in target_order]
            
            # Reorder the data to match 'ZCYX'
            data = np.transpose(data, current_order_indices)
    
        fig = px.imshow(
            data,
            animation_frame=0, 
            binary_string=True, 
            labels=dict(animation_frame="slice", facet_col='channel'), 
            facet_col=1
        )
    # Handle 3D data (no channel dimension)
    elif len(data.shape) == 3:
        fig = px.imshow(
            data,
            animation_frame=0, 
            binary_string=True, 
            labels=dict(animation_frame="slice")
        )
    else:
        raise ValueError("Input must be a 3D or 4D array.")

    # Remove tick labels from axes
    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)
    
    # Set custom size for the plot
    fig.update_layout(**kwargs, title_x=0.5)
        
    # Display the plot using Plotly's show function
    pio.show(fig)

In [ ]:
slice_explorer(data4D, order='ZCYX', title='Cells 3D')

In [ ]:
#| export
@delegates(go.Figure)
def show_volume(values,             # A 3D array of pixel values representing the volume.
                opacity=0.1,        # Opacity level for the surfaces in the volume plot.
                min=None,            # Minimum threshold for the visualization.
                max=None,            # Maximum threshold for the visualization.
                surface_count=5,    # Number of surfaces to display in the volume plot.
                width=800,          # Width of the plotted figure.
                height=600,         # Height of the plotted figure.
                **kwargs            # Additional keyword arguments for Plotly figure customization.
                ):
    """
    Interactive visualization of a 3D volume using Plotly. The function assumes that 'values' is a 3D array representing the volume data.
    """
    # Generate coordinates based on the shape of the values array
    X, Y, Z = np.mgrid[0:values.shape[0], 0:values.shape[1], 0:values.shape[2]]
    if min is None:
        min = values.min()
    if max is None:
        max = values.max()
    
    # Create a Plotly figure with volume data
    fig = go.Figure(data=go.Volume(
        x=X.flatten(),
        y=Y.flatten(),
        z=Z.flatten(),
        value=values.flatten(),
        isomin=min, # Adjust the minimum threshold for visualization
        isomax=max, # Adjust the maximum threshold for visualization
        opacity=opacity, # needs to be small to see through all surfaces
        surface_count=surface_count, # needs to be a large number for good volume rendering
        **kwargs,
    ))
    
    # Update layout with specific width and height
    fig.update_layout(width=width, height=height) # Adjust these values as needed
    
    # Display the figure in a new window
    fig.show()

In [ ]:
show_volume(data[:, 50:150, 50:150])

### Dispatcher

In [ ]:
#| export
def show_biodata(
    data,
    channel_threshold=4,
    max_slices=3,
    volume_mode="slices",
    mip_dim=1,
    **kwargs,
):
    """
    Dispatch to a visualization helper based on input shape.

    Conventions:
    - 2D: YX -> show_image
    - 3D with small first dim: CYX -> show_multichannel
    - 3D otherwise: ZYX -> show_slices or show_mosaic
    - 4D: CZYX -> max projection over Z, then treat as CYX
    """
    if hasattr(data, "as_tensor"):
        x = data.as_tensor().detach().cpu().numpy()
    elif hasattr(data, "detach"):
        x = data.detach().cpu().numpy()
    else:
        x = np.asarray(data)

    if x.ndim == 2:
        return show_image(x, **route_kwargs(show_image, kwargs))

    if x.ndim == 4:
        if not (-x.ndim <= mip_dim < x.ndim):
            raise ValueError(f"`mip_dim` {mip_dim} is out of bounds for shape {x.shape}")
        x = np.max(x, axis=mip_dim)

    if x.ndim == 3:
        if x.shape[0] <= channel_threshold:
            return show_multichannel(x, max_slices=max_slices, **route_kwargs(show_multichannel, kwargs))
        if volume_mode == "mosaic":
            return show_mosaic(x, **route_kwargs(show_mosaic, kwargs))
        if volume_mode == "grid":
            return show_images_grid(x, **route_kwargs(show_images_grid, kwargs))
        if volume_mode == "3D":
            return show_volume(x, **route_kwargs(show_volume, kwargs))
        return show_slices(x, **route_kwargs(show_slices, kwargs))

    raise ValueError(f"Unsupported input shape {x.shape}. Expected 2D, 3D, or 4D data.")

In [ ]:
show_biodata(data4D, cmap='gray', mip_dim=0);

## Animations

In [ ]:
#| export
def show_gif(data,                     # A 3D image stack or 4D image sequence to convert into a GIF.
             path=None,                # Optional filesystem path to save the resulting GIF.
             fps=5,                    # Frames per second for the animation.
             loop=0,                   # Number of times the GIF repeats. 0 means infinite.
             cmap='gray',              # Colormap used when displaying a grayscale stack in Jupyter.
             alt_text='animated gif',  # Alt text for inline notebook display.
             **kwargs):                # Additional keyword arguments for future compatibility.
    """
    Create and display an animated GIF from a 3D stack or 4D image sequence.

    Parameters
    ----------
    data : array-like or tensor
        A stack of frames. Accepted shapes are:
        - (frames, height, width)
        - (frames, height, width, channels)
        - (height, width, channels, frames) if the last axis is 3 or 4.
    path : str or pathlib.Path, optional
        If provided, save the generated GIF to this path.
    fps : int
        Frames per second for the resulting animation.
    loop : int
        Loop count for the GIF. 0 means infinite.
    cmap : str
        Colormap used for grayscale display in notebooks.
    alt_text : str
        Alternative text for the inline HTML image.

    Returns
    -------
    bytes
        The raw GIF bytes.
    """
    # Convert tensors to numpy arrays where possible
    if hasattr(data, 'as_tensor'):
        data = data.as_tensor().detach().cpu().numpy()
    elif hasattr(data, 'detach'):
        try:
            data = data.detach().cpu().numpy()
        except Exception:
            data = np.asarray(data)
    else:
        data = np.asarray(data)

    # Detect frame layout
    if data.ndim == 2:
        raise ValueError('show_gif expects a stack or sequence, not a single 2D image.')
    if data.ndim == 3:
        frames = data
    elif data.ndim == 4:
        if data.shape[-1] in (3, 4):
            frames = data
        elif data.shape[-2] in (3, 4) and data.shape[-1] > 1:
            frames = np.moveaxis(data, -1, 0)
        elif data.shape[-1] == 1:
            frames = data[..., 0]
        else:
            raise ValueError('Ambiguous 4D shape. Provide a frame sequence with channels in the last dimension.')
    else:
        raise ValueError('show_gif supports 3D or 4D arrays only.')

    frames = np.asarray(frames)
    if frames.ndim not in (3, 4):
        raise ValueError('Unsupported frame shape for GIF creation.')

    # Convert values to uint8
    def normalize_frame(x):
        if np.issubdtype(x.dtype, np.floating):
            if x.max() <= 1.0:
                x = np.clip(x * 255.0, 0, 255)
            else:
                x = np.clip(x, 0, 255)
            return x.astype(np.uint8)
        if x.dtype == np.bool_ or x.max() <= 1:
            return (x.astype(np.uint8) * 255)
        return x.astype(np.uint8)

    frames_uint8 = normalize_frame(frames)

    if frames_uint8.ndim == 4 and frames_uint8.shape[-1] == 1:
        frames_uint8 = frames_uint8[..., 0]

    if frames_uint8.ndim == 3 and cmap != 'gray':
        cmap_obj = plt.get_cmap(cmap)
        frames_uint8 = np.stack([
            (cmap_obj(frame.astype(np.float32) / 255.0)[..., :3] * 255).astype(np.uint8)
            for frame in frames_uint8
        ], axis=0)

    # Create GIF bytes
    buffer = io.BytesIO()

    def _write_gif_bytes(output):
        try:
            import imageio
            with imageio.get_writer(output, mode='I', format='GIF', duration=1.0 / fps, loop=loop) as writer:
                for frame in frames_uint8:
                    writer.append_data(frame)
            return
        except Exception:
            pass

        try:
            from PIL import Image
            pil_frames = []
            for frame in frames_uint8:
                if frame.ndim == 2:
                    pil_frames.append(Image.fromarray(frame, mode='L'))
                elif frame.shape[2] == 4:
                    pil_frames.append(Image.fromarray(frame, mode='RGBA'))
                else:
                    pil_frames.append(Image.fromarray(frame, mode='RGB'))
            pil_frames[0].save(output, format='GIF', save_all=True, append_images=pil_frames[1:], duration=int(round(1000.0 / fps)), loop=loop)
            return
        except Exception as exc:
            raise RuntimeError('Unable to write GIF. Install imageio or pillow to use show_gif.') from exc

    _write_gif_bytes(buffer)
    gif_bytes = buffer.getvalue()

    if path is not None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(gif_bytes)

    try:
        from IPython.display import HTML, display
        data64 = base64.b64encode(gif_bytes).decode('ascii')
        html = HTML(f'<img src="data:image/gif;base64,{data64}" alt="{alt_text}" style="max-width:100%; height:auto;" />')
        display(html)
    except Exception:
        return gif_bytes

In [ ]:
show_gif(data, "./data_examples/output.gif")

## Plots

In [ ]:
#| export
colorlist = [
            'dimgray', 'firebrick', 'goldenrod', 'teal',
            '#102e42', 'darkslateblue', '#1c5175',
            'cadetblue', 'steelblue'
        ]

In [ ]:
#| export
bonsai_style = {
    'font.family': 'sans-serif',
    'font.size': 10,
    'text.color': 'black',
    'axes.labelsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'axes.titlesize': 16,
    'axes.grid': True,
    'axes.axisbelow': False,
    'grid.alpha': 0.25,
    'grid.linestyle': '-',
    'grid.color': 'gray',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'xtick.labelcolor': 'black',
    'ytick.labelcolor': 'black',
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': True,
    'axes.spines.bottom': True,
    'figure.facecolor': 'white',
    'savefig.facecolor': 'white',
    'lines.color': 'black',
    'axes.prop_cycle': plt.cycler(
        'color',
        colorlist
    )
}

In [ ]:
#| export

def plot_metrics(
    learn,
    style=bonsai_style,
    minimize_metrics=False,
    orientation="vertical",
    **subplots_kwargs
):
    """
    Plot losses and metrics from a fastai Recorder.

    Parameters
    ----------
    learn : fastai Learner
    style : dict or None
        Temporary matplotlib rcParams style.
    minimize_metrics : bool
        If True, metrics are minimized when determining the best epoch.
        Otherwise, metrics are maximized.
    orientation : {"vertical", "horizontal"}
        Arrange plots vertically or horizontally.
    **subplots_kwargs
        Additional keyword arguments passed to plt.subplots().
    """

    if orientation not in ("vertical", "horizontal"):
        raise ValueError(
            "orientation must be either 'vertical' or 'horizontal'"
        )

    ctx = plt.rc_context(style) if style else nullcontext()

    with ctx:

        recorder = learn.recorder
        metric_names = recorder.metric_names[1:-1]

        df = pd.DataFrame(
            recorder.values,
            columns=metric_names
        )

        epochs = np.arange(0, len(df))

        loss_cols = [
            c for c in df.columns
            if "loss" in c
        ]

        metric_cols = [
            c for c in df.columns
            if "loss" not in c
        ]

        n_plots = 1 + len(metric_cols)

        # --------------------------------
        # Subplot layout
        # --------------------------------

        if orientation == "vertical":
            defaults = dict(
                figsize=(7, 4 * n_plots),
                sharex=True
            )
            nrows, ncols = n_plots, 1

        else:
            defaults = dict(
                figsize=(7 * n_plots, 4),
                sharey=False
            )
            nrows, ncols = 1, n_plots

        defaults.update(subplots_kwargs)

        fig, axes = plt.subplots(
            nrows,
            ncols,
            **defaults
        )

        axes = np.atleast_1d(axes)

        # --------------------------------
        # Loss plot
        # --------------------------------

        ax = axes[0]

        for col in loss_cols:
            ax.plot(
                epochs,
                df[col],
                marker="o",
                label=col
            )

        if "valid_loss" in df:
            best_epoch = df["valid_loss"].idxmin()

            ax.axvline(
                best_epoch,
                linestyle="--",
                alpha=0.6
            )

            ax.scatter(
                best_epoch,
                df["valid_loss"][best_epoch],
                s=80
            )

        ax.set_title("Training and Validation Loss")
        ax.set_ylabel("Loss")
        ax.set_xlabel("Epoch")
        ax.legend()
        ax.grid(True)

        # --------------------------------
        # Metrics
        # --------------------------------

        metric_color = (plt.rcParams["axes.prop_cycle"].by_key()["color"][1])

        for i, metric in enumerate(metric_cols):

            ax = axes[i + 1]

            ax.plot(
                epochs,
                df[metric],
                marker="o",
                color=metric_color,
                label=metric
            )

            if minimize_metrics:
                best_epoch = df[metric].idxmin()
            else:
                best_epoch = df[metric].idxmax()

            ax.axvline(
                best_epoch,
                linestyle="--",
                alpha=0.6,
                color="black"
            )

            ax.scatter(
                best_epoch,
                df[metric][best_epoch],
                s=80,
                color="black"
            )

            ax.set_title(metric)
            ax.set_ylabel(metric)
            ax.set_xlabel("Epoch")
            ax.grid(True)

        plt.tight_layout()
        plt.show()

In [ ]:
from types import SimpleNamespace

In [ ]:
epochs = 10

fake_values = [
    [
        train_loss,
        valid_loss,
        accuracy,
        f1
    ]
    for train_loss, valid_loss, accuracy, f1 in zip(
        np.linspace(1.2, 0.2, epochs),                    # train_loss
        [1.1, 0.9, 0.8, 0.7, 0.55, 0.5, 0.52, 0.48, 0.5, 0.51],  # valid_loss
        np.linspace(0.55, 0.93, epochs),                 # accuracy
        np.linspace(0.50, 0.90, epochs),                 # f1
    )
]

fake_recorder = SimpleNamespace(
    metric_names=[
        "epoch",
        "train_loss",
        "valid_loss",
        "accuracy",
        "f1",
        "time"
    ],
    values=fake_values
)

fake_learn = SimpleNamespace(
    recorder=fake_recorder
)

plot_metrics(fake_learn, figsize=(6, 12));

In [ ]:
#| export
def plot_intensity_histogram(
    image,
    threshold=None,
    mode=None,          # None, "greater", "lower"
    bins=256,
    per_channel=False,
    color="steelblue",
    density=False,
    int_ticks=True,
    int_y_ticks=False,
    style=bonsai_style,
    **subplots_kwargs
):
    """
    Plot intensity histogram of an image.

    Parameters
    ----------
    image : array-like
        Input image.

    threshold : float or None
        Threshold value for filtering.

    mode : str or None
        Filtering mode:
        - None
        - "greater"
        - "lower"

    bins : int
        Number of histogram bins.

    per_channel : bool
        Plot RGB channels separately.

    color : str
        Histogram color for grayscale/combined mode.

    density : bool
        Normalize histogram.

    int_ticks : bool
        Force integer x-axis ticks.

    int_y_ticks : bool
        Force integer y-axis ticks.

    style : dict or None
        Temporary matplotlib rcParams style.

    **subplots_kwargs
        Additional kwargs passed to plt.subplots().
    """

    ctx = plt.rc_context(style) if style else nullcontext()

    with ctx:

        image = np.asarray(image)

        def filter_values(values):
            if threshold is None or mode is None:
                return values

            if mode == "greater":
                return values[values > threshold]

            elif mode == "lower":
                return values[values < threshold]

            else:
                raise ValueError(
                    "mode must be None, 'greater', or 'lower'"
                )

        defaults = dict(figsize=(6, 4))
        defaults.update(subplots_kwargs)

        fig, ax = plt.subplots(**defaults)

        # ---- Grayscale ----

        if image.ndim == 2:

            values = filter_values(image.ravel())

            ax.hist(
                values,
                bins=bins,
                color=color,
                alpha=0.9,
                density=density
            )

            ax.set_title("Grayscale Intensity Histogram")

        # ---- RGB ----

        elif image.ndim == 3 and image.shape[2] == 3:

            if per_channel:

                colors = ["red", "green", "blue"]

                for i, c in enumerate(colors):

                    values = filter_values(image[..., i].ravel())

                    ax.hist(
                        values,
                        bins=bins,
                        color=c,
                        alpha=0.5,
                        density=density,
                        label=f"{c.upper()} channel"
                    )

                ax.legend()

                ax.set_title(
                    "RGB Intensity Histogram (Per Channel)"
                )

            else:

                values = filter_values(image.ravel())

                ax.hist(
                    values,
                    bins=bins,
                    color="gray",
                    alpha=0.7,
                    density=density
                )

                ax.set_title(
                    "RGB Combined Intensity Histogram"
                )

        else:
            raise ValueError("Unsupported image shape")

        ax.set_xlabel("Intensity")
        ax.set_ylabel("Frequency")

        # ---- Integer ticks ----

        if int_ticks:
            ax.xaxis.set_major_locator(
                MaxNLocator(integer=True)
            )

        if int_y_ticks:
            ax.yaxis.set_major_locator(
                MaxNLocator(integer=True)
            )

        plt.tight_layout()
        plt.show()

In [ ]:
# ---------------- Grayscale image ----------------

gray_image = np.random.normal(
    loc=120,
    scale=30,
    size=(256, 256)
).clip(0, 255).astype(np.uint8)

plot_intensity_histogram(gray_image, bins=128)


# ---------------- Thresholded histogram ----------------

plot_intensity_histogram(
    gray_image,
    threshold=100,
    bins=50,
    mode="greater",
    color="goldenrod"
)



# ---------------- Disable custom style ----------------

plot_intensity_histogram(
    gray_image,
    style=None
)

In [ ]:
#| export
def plot_roc_curve_with_std(
    y_probs_folds,
    y_true_folds,
    n_points=200,
    show_folds=False,
    title="ROC curve with cross-validation"
):
    """
    Plot ROC curves across CV folds with mean ROC and ﷿﷿﷿﷿﷿﷿1 std band.

    Parameters
    ----------
    y_probs_folds : list of array-like
        Predicted probabilities for the positive class per fold.
    y_true_folds : list of array-like
        Ground truth labels per fold.
    n_points : int
        Resolution of interpolated ROC curve.
    show_folds : bool
        Whether to display individual fold curves.
    title : str
        Plot title.
    """

    fpr_grid = np.linspace(0, 1, n_points)

    tprs = []
    aucs = []

    fig, ax = plt.subplots(figsize=(6, 6))

    # ---------------------------------------------------------------------
    # Per-fold ROC
    # ---------------------------------------------------------------------
    for fold, (y_prob, y_true) in enumerate(zip(y_probs_folds, y_true_folds)):

        roc = RocCurveDisplay.from_predictions(
            y_true,
            y_prob,
            ax=ax,
            curve_kwargs={
                "alpha": 0.25 if show_folds else 0.0,
                "lw": 1
            }
        )

        # Interpolate TPR onto common grid
        tpr_interp = np.interp(fpr_grid, roc.fpr, roc.tpr)
        tpr_interp[0] = 0.0

        tprs.append(tpr_interp)
        aucs.append(roc.roc_auc)

        # Hide legend entries if not requested
        if not show_folds:
            roc.line_.set_label("_nolegend_")

    # ---------------------------------------------------------------------
    # Aggregate statistics
    # ---------------------------------------------------------------------
    tprs = np.array(tprs)

    mean_tpr = tprs.mean(axis=0)
    mean_tpr[-1] = 1.0

    std_tpr = tprs.std(axis=0)

    mean_auc = np.mean(aucs)
    std_auc = np.std(aucs)

    # ---------------------------------------------------------------------
    # Mean ROC curve
    # ---------------------------------------------------------------------
    ax.plot(
        fpr_grid,
        mean_tpr,
        color="red",
        lw=2,
        label=f"Mean ROC (AUC = {mean_auc:.2f} ﷿﷿﷿﷿﷿﷿ {std_auc:.2f})"
    )

    # ---------------------------------------------------------------------
    # Confidence band
    # ---------------------------------------------------------------------
    ax.fill_between(
        fpr_grid,
        np.maximum(mean_tpr - std_tpr, 0),
        np.minimum(mean_tpr + std_tpr, 1),
        color="grey",
        alpha=0.2,
        label="﷿﷿﷿﷿﷿﷿1 std. dev."
    )

    # ---------------------------------------------------------------------
    # Final styling
    # ---------------------------------------------------------------------
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend(loc="lower right")

    plt.show()

In [ ]:
#| export
def plot_roc_curve_with_std_interactive(
    y_probs_folds,
    y_true_folds,
    n_points=200,
    show_folds=True,
    title="ROC curve with cross-validation"
):

    fpr_grid = np.linspace(0, 1, n_points)

    tprs = []
    aucs = []

    fig = go.Figure()

    # ------------------------------------------------------------
    # Per-fold ROC
    # ------------------------------------------------------------
    for i, (y_prob, y_true) in enumerate(zip(y_probs_folds, y_true_folds)):

        fpr, tpr, _ = roc_curve(y_true, y_prob)
        roc_auc = auc(fpr, tpr)

        tpr_interp = np.interp(fpr_grid, fpr, tpr)
        tpr_interp[0] = 0.0

        tprs.append(tpr_interp)
        aucs.append(roc_auc)

        if show_folds:
            fig.add_trace(go.Scatter(
                x=fpr,
                y=tpr,
                mode="lines",
                name=f"Fold {i+1} (AUC={roc_auc:.2f})",
                opacity=0.4
            ))

    tprs = np.array(tprs)

    # ------------------------------------------------------------
    # Mean + std
    # ------------------------------------------------------------
    mean_tpr = tprs.mean(axis=0)
    mean_tpr[-1] = 1.0

    std_tpr = tprs.std(axis=0)

    mean_auc = np.mean(aucs)
    std_auc = np.std(aucs)

    upper = np.minimum(mean_tpr + std_tpr, 1)
    lower = np.maximum(mean_tpr - std_tpr, 0)

    # Mean ROC
    fig.add_trace(go.Scatter(
        x=fpr_grid,
        y=mean_tpr,
        mode="lines",
        name=f"Mean ROC (AUC={mean_auc:.2f} ﷿﷿﷿﷿﷿﷿ {std_auc:.2f})",
        line=dict(width=3, color="red")
    ))

    # Std band
    fig.add_trace(go.Scatter(
        x=np.concatenate([fpr_grid, fpr_grid[::-1]]),
        y=np.concatenate([upper, lower[::-1]]),
        fill="toself",
        fillcolor="rgba(128,128,128,0.2)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        name="﷿﷿﷿﷿﷿﷿1 std. dev."
    ))

    # Chance line
    fig.add_trace(go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        line=dict(dash="dash", color="gray"),
        name="Chance"
    ))

    # ------------------------------------------------------------
    # Styling: black axes + centered title
    # ------------------------------------------------------------
    fig.update_layout(
        title=dict(
            text=title,
            x=0.5,              # center title
            xanchor="center"
        ),
            xaxis=dict(
            range=[0, 1],
            autorange=False,
            title="False Positive Rate",
            showline=True,
            linecolor="black",
            mirror=True,
            ticks="outside",
            scaleanchor="y",
            scaleratio=1
        ),
        yaxis=dict(
            range=[0, 1],
            autorange=False,
            title="True Positive Rate",
            showline=True,
            linecolor="black",
            mirror=True,
            ticks="outside"
        ),
        width=700,
        height=700,
        template="plotly_white",
        legend=dict(orientation="v")
    )

    fig.show()

    return fig

Plot of the ROC curve for the data after applying cross-validation and training in order to visualize the standard deviation to compare between splits.

In [ ]:
from sklearn.svm import SVC
from sklearn.datasets import load_iris
from sklearn import svm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import RocCurveDisplay, auc, roc_curve

In [ ]:
# -----------------------------------------------------------------------------
# 1. Load and preprocess data
# -----------------------------------------------------------------------------
iris = load_iris()
X, y = iris.data, iris.target

# Keep only two classes (binary classification)
mask = y != 2
X, y = X[mask], y[mask]

n_samples, n_features = X.shape

# -----------------------------------------------------------------------------
# 2. Feature augmentation (noise injection for high-dimensional setting)
# -----------------------------------------------------------------------------
rng = np.random.RandomState(0)
noise = rng.randn(n_samples, 300 * n_features)

X = np.concatenate([X, noise], axis=1)

# -----------------------------------------------------------------------------
# 3. Cross-validation setup
# -----------------------------------------------------------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# -----------------------------------------------------------------------------
# 4. Model
# -----------------------------------------------------------------------------
model = SVC(kernel="linear", probability=True, random_state=0)

# -----------------------------------------------------------------------------
# 5. Cross-validated predictions
# -----------------------------------------------------------------------------
y_probs_folds = []
y_true_folds = []

for train_idx, test_idx in cv.split(X, y):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model.fit(X_train, y_train)

    probs = model.predict_proba(X_test)[:, 1]

    y_probs_folds.append(probs)
    y_true_folds.append(y_test)

# -----------------------------------------------------------------------------
# 6. Evaluation
# -----------------------------------------------------------------------------
plot_roc_curve_with_std(
    y_probs_folds,
    y_true_folds
)

plot_roc_curve_with_std_interactive(
    y_probs_folds,
    y_true_folds
);

In [ ]:
#| export
def format_sig(value):
    """
    Format numbers with two significant digits.
    """
    if value == 0:
        return "0"
    elif abs(value) < 0.01 or abs(value) > 100:
        return f"{value:.2e}"  # 2 significant digits in scientific notation
    else:
        return f"{value:.2g}"  # Standard notation with 2 significant figures

In [ ]:
#| export
def plot_histogram_and_kde(
    data,
    stats,
    bw_method=0.3,
    fn_name='',
    style=bonsai_style,
    **subplots_kwargs
):
    """
    Plot histogram and KDE with key statistics marked.

    Parameters
    ----------
    data : array-like
        Input values.

    stats : dict
        Dictionary containing summary statistics.

    bw_method : float or str
        Bandwidth method for gaussian_kde.

    fn_name : str
        Optional subtitle/title suffix.

    style : dict or None
        Temporary matplotlib rcParams style.

    **subplots_kwargs
        Additional kwargs passed to plt.subplots().
    """

    ctx = plt.rc_context(style) if style else nullcontext()

    with ctx:

        data = np.asarray(data)

        kde = gaussian_kde(data, bw_method=bw_method)

        x = np.linspace(
            np.min(data),
            np.max(data),
            1000
        )

        y = kde(x)

        defaults = dict(figsize=(8, 6))
        defaults.update(subplots_kwargs)

        fig, ax = plt.subplots(**defaults)

        gauss_color = 'lightslategrey'

        # ---- Histogram ----

        ax.hist(
            data,
            bins=30,
            density=True,
            color='darkgray',
            edgecolor=None,
            alpha=0.5
        )

        # ---- KDE ----

        ax.plot(
            x,
            y,
            color=gauss_color,
            lw=2
        )

        ax.fill_between(
            x,
            y,
            color=gauss_color,
            alpha=0.3
        )

        # ---- Statistics lines ----

        ax.axvline(
            stats["Mean"],
            color='crimson',
            linestyle='--',
            linewidth=1.5,
            label=f'Mean: {format_sig(stats["Mean"])}'
        )

        ax.axvline(
            stats["Median"],
            color='steelblue',
            linestyle='--',
            linewidth=1.5,
            label=f'Median: {format_sig(stats["Median"])}'
        )

        ax.axvline(
            stats["Q1"],
            color='firebrick',
            linestyle=':',
            linewidth=1.5,
            label=f'Q1: {format_sig(stats["Q1"])}'
        )

        ax.axvline(
            stats["Q3"],
            color='firebrick',
            linestyle=':',
            linewidth=1.5,
            label=f'Q3: {format_sig(stats["Q3"])}'
        )

        # ---- Text annotations ----

        ax.text(
            stats["Mean"] + stats["Standard Deviation"],
            0.1,
            f'Std Dev: {format_sig(stats["Standard Deviation"])}',
            color='black',
            fontsize=10
        )

        ax.text(
            stats["Min"],
            0.02,
            f'Min: {format_sig(stats["Min"])}',
            color='black',
            fontsize=10,
            ha='center'
        )

        ax.text(
            stats["Max"],
            0.02,
            f'Max: {format_sig(stats["Max"])}',
            color='black',
            fontsize=10,
            ha='center'
        )

        # ---- Labels ----

        ax.set_title(
            f"Combined Histogram and KDE with Statistics\n{fn_name}"
        )

        ax.set_xlabel("Value")
        ax.set_ylabel("Density")

        ax.legend()

        plt.tight_layout()
        plt.show()

In [ ]:
#| export
def display_statistics_table(stats, fn_name='', as_dataframe=True):
    """
    Display a table of the key statistics.
    """
    if as_dataframe:
        # Convert statistics to a DataFrame and display
        df = pd.DataFrame.from_dict(stats, orient='index', columns=['Value'])
        df.index.name = f"{fn_name}"
        display(df)
    else:
        fig, ax = plt.subplots(figsize=(5, 2))
        ax.axis("off")
        
        # Header title row
        table_data = [[f"{fn_name}", ""]]
        table_data += [[key, format_sig(value)] for key, value in stats.items()]
        
        table = ax.table(cellText=table_data, colLabels=["Statistic", "Value"], cellLoc="center", loc="center")
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1.2, 1.2)
        
        # Style for header row
        header = table[0, 0]
        header.set_fontsize(12)
        header.set_text_props(weight="bold")
        plt.show()

In [ ]:
from numpy.random import standard_normal
from bonsai.inference import calculate_statistics

In [ ]:

a = standard_normal(1000)

stats = calculate_statistics(a)

plot_histogram_and_kde(a, stats)

In [ ]:
#| export
def plot_dist(
    values,
    marginal="box",
    nbins=None,
    color=colorlist,
    title=None,
    labels=None,
    opacity=0.65,
    barmode="overlay",
    markerline_width=0.5,
    show_kde=False,
):
    """
    Plot one or more distributions with a Matplotlib-like Plotly layout.

    Parameters
    ----------
    values : array-like or list of array-like
        One distribution or multiple distributions.
    marginal : str or None
        One of {"box", "violin", None}.
    nbins : int or None
        Number of histogram bins.
    color : str or list of str or None
        One color, one per distribution, or None to use Plotly defaults.
    title : str or None
        Optional plot title.
    labels : list of str or None
        Names for each distribution.
    opacity : float
        Bar opacity when plotting multiple distributions.
    barmode : str
        Plotly bar mode, usually "overlay" or "group".
    show_kde : bool
        If True, overlay KDE curves on the same density axis.
    """
    if isinstance(values, (list, tuple)) and len(values) > 0 and hasattr(values[0], "__len__"):
        dists = values
    else:
        dists = [values]

    if labels is None:
        labels = [f"dist_{i}" for i in range(len(dists))]

    if len(labels) != len(dists):
        raise ValueError("`labels` must have the same length as `values`.")

    clean_dists = []
    frames = []
    for vals, label in zip(dists, labels):
        x = np.asarray(vals).ravel()
        x = x[~np.isnan(x)]
        clean_dists.append(x)
        frames.append(pd.DataFrame({"values": x, "dist": label}))

    df = pd.concat(frames, ignore_index=True)

    hist_kwargs = dict(
        data_frame=df,
        x="values",
        color="dist",
        marginal=marginal,
        hover_data=df.columns,
        nbins=nbins,
        histnorm="probability density",
        template="simple_white",
        title=title,
        barmode=barmode,
        opacity=opacity,
    )

    colors = None
    if color is not None:
        if isinstance(color, str):
            colors = [color] * len(dists)
        else:
            colors = list(color)
            if len(colors) >= len(dists):
                colors = colors[:len(dists)]
            if len(colors) < len(dists):
                raise ValueError("`color` must be a string or a list with at least one color per distribution.")
        hist_kwargs["color_discrete_sequence"] = colors

    fig = px.histogram(**hist_kwargs)

    fig.update_traces(marker=dict(line=dict(color="white", width=markerline_width)))

    if show_kde:
        import plotly.figure_factory as ff

        kde_colors = colors
        if kde_colors is None:
            kde_colors = px.colors.qualitative.Plotly[:len(clean_dists)]

        kde_fig = ff.create_distplot(
            [x.tolist() for x in clean_dists],
            group_labels=labels,
            colors=kde_colors,
            show_hist=False,
            show_rug=False,
        )

        for trace in kde_fig.data:
            fig.add_trace(trace)

    fig.update_layout(
        font=dict(family="DejaVu Sans, Arial, sans-serif", size=12, color="black"),
        plot_bgcolor="white",
        paper_bgcolor="white",
        legend_title_text="",
    )

    fig.update_xaxes(
        showline=True,
        linewidth=1,
        linecolor="black",
        ticks="outside",
        tickwidth=1,
        tickcolor="black",
        showgrid=False,
    )

    fig.update_yaxes(
        title_text="Density",
        showline=True,
        linewidth=1,
        linecolor="black",
        ticks="outside",
        tickwidth=1,
        tickcolor="black",
        showgrid=False,
    )

    fig.show()
    return fig


In [ ]:
b = standard_normal(1000) + 2
plot_dist([a,b], labels=["Distribution A", "Distribution B"], show_kde=True, marginal="box");


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()